In [ ]:
# Install dependencies
!pip install kaggle
!pip install opencv-python-headless
!pip install torch torchvision torchaudio
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 70.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 13.7 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659448 sha256=132de3d79049110dfa992196a8f777171eb04353f7c432e29a11fbf8672c1efe
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [ ]:
# Move the uploaded kaggle.json to the correct folder
!mkdir -p ~/.kaggle
!mv /content/kaggle.json ~/.kaggle/

# Set file permissions
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mksaad/wider-face-a-face-detection-benchmark")

print("Path to dataset files:", path)

100%|██████████| 3.41G/3.41G [02:38<00:00, 23.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mksaad/wider-face-a-face-detection-benchmark/versions/4


In [ ]:
import shutil

# Define the original and new locations
original_path = '/root/.cache/kagglehub/datasets/mksaad/wider-face-a-face-detection-benchmark/versions/4'
new_path = '/content/wider_face'

# Move the dataset to /content/wider_face
shutil.move(original_path, new_path)

# Check if the dataset is moved correctly
import os
print("Files in the new dataset location:", os.listdir(new_path))

Files in the new dataset location: ['train.py', 'download_model.cpython-36.pyc', 'wider_face_split', 'eval.py', 'download_model.py', 'filelog.log', 'test_detection.py', 'wider_val', 'wider_train', 'wider_face_dataset.py', 'blacklist.txt', 'demo.py', 'WIDER_val', 'WIDER_train']


In [ ]:
# List the contents of the 'wider_face_split' subdirectory
split_subdir = '/content/wider_face/wider_face_split/wider_face_split'
os.listdir(split_subdir)

['wider_face_test_filelist.txt',
 'wider_face_val_bbx_gt.txt',
 'wider_face_train_bbx_gt.txt',
 'wider_face_test.mat',
 'readme.txt',
 'wider_face_val.mat',
 'wider_face_train.mat']

In [ ]:
# Read the training annotation file
train_annotation_file = '/content/wider_face/wider_face_split/wider_face_split/wider_face_train_bbx_gt.txt'

with open(train_annotation_file, 'r') as f:
    train_annotations = f.readlines()

# Print the first few lines to understand the format
train_annotations[:10]

['0--Parade/0_Parade_marchingband_1_849.jpg\n',
 '1\n',
 '449 330 122 149 0 0 0 0 0 0 \n',
 '0--Parade/0_Parade_Parade_0_904.jpg\n',
 '1\n',
 '361 98 263 339 0 0 0 0 0 0 \n',
 '0--Parade/0_Parade_marchingband_1_799.jpg\n',
 '21\n',
 '78 221 7 8 2 0 0 0 0 0 \n',
 '78 238 14 17 2 0 0 0 0 0 \n']

In [ ]:
import os
import cv2 # Import for reading image dimensions

def convert_to_yolo_format(annotation_lines_for_one_image, image_path_actual):
    """Convert the WIDER FACE annotation to YOLO format."""
    # Get actual image dimensions
    img = cv2.imread(image_path_actual)
    if img is None:
        # Some images might be corrupted or missing, we will skip them during processing
        return None, []
    height, width, _ = img.shape
    # image_size expects (width, height)
    image_size = (width, height)

    yolo_labels = []

    # The first line is the relative image path, the second is num_faces
    # The function expects 'annotation_lines_for_one_image' to contain:
    # [relative_image_path_str, num_faces_str, bbox1_str, bbox2_str, ...]
    img_name = annotation_lines_for_one_image[0] # This is already the image path string
    num_faces = int(annotation_lines_for_one_image[1])

    # Process each bounding box from the remaining lines
    # The bounding box data starts from index 2
    for i in range(num_faces):
        box_data_str = annotation_lines_for_one_image[2 + i]
        box_data = list(map(int, box_data_str.split()))
        x_min, y_min, box_width, box_height = box_data[:4]

        # Convert to YOLO format (center_x, center_y, width, height)
        x_center = (x_min + box_width / 2) / width
        y_center = (y_min + box_height / 2) / height
        width_norm = box_width / width
        height_norm = box_height / height

        # Class 0 for face detection
        yolo_labels.append(f"0 {x_center} {y_center} {width_norm} {height_norm}")

    return img_name, yolo_labels

# Define base directories
wider_face_root = '/content/wider_face'
output_images_root = os.path.join(wider_face_root, 'images')
output_labels_root = os.path.join(wider_face_root, 'labels')

# Create necessary directories for train and val splits
os.makedirs(os.path.join(output_images_root, 'train'), exist_ok=True)
os.makedirs(os.path.join(output_images_root, 'val'), exist_ok=True)
os.makedirs(os.path.join(output_labels_root, 'train'), exist_ok=True)
os.makedirs(os.path.join(output_labels_root, 'val'), exist_ok=True)

def process_wider_annotations(annotation_file_path, original_image_base_path, dest_image_base_path, dest_label_base_path):
    with open(annotation_file_path, 'r') as f:
        annotations = f.readlines()

    current_line_idx = 0
    skipped_images = 0
    while current_line_idx < len(annotations):
        image_relative_path = annotations[current_line_idx].strip()
        current_line_idx += 1

        num_faces_str = annotations[current_line_idx].strip()
        num_faces = int(num_faces_str)
        current_line_idx += 1

        image_annotations_data = [image_relative_path, num_faces_str]
        for _ in range(num_faces):
            image_annotations_data.append(annotations[current_line_idx].strip())
            current_line_idx += 1

        # Construct full path to original image
        original_img_path = os.path.join(original_image_base_path, image_relative_path)

        # Construct destination paths (image and label)
        # Create subdirectories in destination if they exist in relative path
        dest_img_subdir = os.path.dirname(image_relative_path)
        dest_img_dir = os.path.join(dest_image_base_path, dest_img_subdir)
        os.makedirs(dest_img_dir, exist_ok=True)
        dest_img_path = os.path.join(dest_img_dir, os.path.basename(image_relative_path))

        dest_label_subdir = os.path.dirname(image_relative_path)
        dest_label_dir = os.path.join(dest_label_base_path, dest_label_subdir)
        os.makedirs(dest_label_dir, exist_ok=True)
        dest_label_file_name = os.path.basename(image_relative_path).rsplit('.', 1)[0] + '.txt'
        dest_label_path = os.path.join(dest_label_dir, dest_label_file_name)

        # Create symbolic link for image to avoid duplicating large image files
        if not os.path.exists(original_img_path): # Check if the source image exists
            print(f"Warning: Original image not found: {original_img_path}. Skipping labels for this image.")
            skipped_images += 1
            continue

        if not os.path.exists(dest_img_path): # Create symlink only if it doesn't exist
            os.symlink(original_img_path, dest_img_path)

        # Convert annotations to YOLO format
        # Pass the actual full path to the image so convert_to_yolo_format can read its dimensions
        converted_img_name, yolo_labels = convert_to_yolo_format(image_annotations_data, original_img_path)

        # If image was skipped due to inability to read, skip label saving too
        if converted_img_name is None:
            continue

        # Save YOLO labels
        with open(dest_label_path, 'w') as label_f:
            label_f.write("\n".join(yolo_labels))
    print(f"Processed annotations and images for: {annotation_file_path}. Skipped {skipped_images} images.")

# Process training data
train_annotation_file = os.path.join(wider_face_root, 'wider_face_split', 'wider_face_split', 'wider_face_train_bbx_gt.txt')
original_train_images_path = os.path.join(wider_face_root, 'WIDER_train', 'WIDER_train', 'images') # Based on kernel state 'images_dir'
dest_train_images_path = os.path.join(output_images_root, 'train')
dest_train_labels_path = os.path.join(output_labels_root, 'train')
process_wider_annotations(train_annotation_file, original_train_images_path, dest_train_images_path, dest_train_labels_path)

# Process validation data
val_annotation_file = os.path.join(wider_face_root, 'wider_face_split', 'wider_face_split', 'wider_face_val_bbx_gt.txt')
original_val_images_path = os.path.join(wider_face_root, 'WIDER_val', 'WIDER_val', 'images') # Assuming similar structure
dest_val_images_path = os.path.join(output_images_root, 'val')
dest_val_labels_path = os.path.join(output_labels_root, 'val')
process_wider_annotations(val_annotation_file, original_val_images_path, dest_val_images_path, dest_val_labels_path)

print("Annotations converted to YOLO format and saved, and images symlinked.")


Processed annotations and images for: /content/wider_face/wider_face_split/wider_face_split/wider_face_train_bbx_gt.txt. Skipped 0 images.
Processed annotations and images for: /content/wider_face/wider_face_split/wider_face_split/wider_face_val_bbx_gt.txt. Skipped 0 images.
Annotations converted to YOLO format and saved, and images symlinked.


In [ ]:
!pip install -U ultralytics supervision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 20.3 MB/s eta 0:00:00


In [ ]:
import yaml
import os

dataset_yaml = {
    "train": "/content/wider_face/images/train",
    "val": "/content/wider_face/images/val",
    "names": ["face"]
}

os.makedirs('data', exist_ok=True)
with open("data/face.yaml","w") as f:
    yaml.dump(dataset_yaml, f)

In [ ]:
from ultralytics import YOLO

# model = YOLO("yolo26n.pt")      # Nano variant
model = YOLO("yolo26s.pt")         # Small variant (better quality than nano)

results = model.train(
    data="data/face.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="runs/yolo26_face",
    name="exp1"
)

Ultralytics 8.4.17 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/face.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp12, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, 